# Tree Structure and Recursive Models


```{contents}
:local:
:depth: 2
```


```{index} Tree Structure and Traversal
```

<span id="tree-structure-and-traversal"></span>


```{index} Tree Vocabulary
```

## Tree Vocabulary




```{index} tree; root; leaf; subtree
```


A **tree** is a collection of nodes connected by parent-child relationships. The **root** is the starting node. A **leaf** has no children. A **subtree** is a node together with everything below it.

Many real systems have this shape: folders contain folders, supervisors manage teams, expressions contain subexpressions, and web pages link to lower-level navigation sections.


In [1]:
using System;

var root = new TreeNode("Courses");
root.Left = new TreeNode("Programming");
root.Right = new TreeNode("Databases");
root.Left.Left = new TreeNode("C#");
root.Left.Right = new TreeNode("Python");

Console.WriteLine(root.Value);
Console.WriteLine(root.Left.Value);
Console.WriteLine(root.Left.Left.Value);

class TreeNode
{
    public string Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(string value)
    {
        Value = value;
    }
}


Courses
Programming
C#


This first representation uses `Left` and `Right` references, so it is a **binary tree**. The values could represent labels, numbers, records, or any other data.


**Common mistake.** Assigning `root.Left.Left = new TreeNode("C#")` before `root.Left` exists throws `NullReferenceException`, because there is no node to hang the child on yet. Build from the root downward, or use a constructor that takes the children, as the final section does.


A tree models a hierarchy with one route from the root to each node. In a nonempty tree, every node except the root has exactly one parent. There are no cycles, and all nodes belong to the same connected structure. A structure with shared children or a link back to an ancestor needs graph algorithms instead of these tree algorithms.

A company's departments form a general tree: a manager can have more than two direct reports. `Left` and `Right` deliberately restrict the example to two children. The names indicate positions, not a numerical ordering rule. A binary tree can contain unsorted labels or numbers.

For the course catalog example, draw each reference as an edge:

```text
        Courses
       /       \
Programming   Databases
   /    \
 C#    Python
```

`Courses` has depth zero, `Programming` has depth one, and `C#` has depth two. There are five nodes and four edges. Every non-root node contributes its one incoming edge, so a nonempty tree with n nodes has n - 1 edges. An empty tree has no root and zero edges.

Before running the next example, identify all leaves. A node with only one child is an internal node, not a leaf. Two labels can be equal and still belong to different nodes; a node is an object, while its label is stored data.

A recursive method needs two promises. First, it must know when to stop. Second, every recursive call must move closer to that stopping point.

Chapter 5's supporting reading {doc}`Recursive Methods <../05_methods/0506_recursive_methods>` introduced these promises with a countdown and a sum; reread it if the base case and progress argument are new. Recursion is not magic; it is ordinary method calls plus the call stack. The hard part is choosing a smaller version of the problem that has the same shape as the original problem.


```{index} Base Case and Recursive Case
```

## Base Case and Recursive Case




```{index} Base Case and Recursive Case
```

A **base case** returns an answer directly. A **recursive case** calls the same method on a smaller input and uses that result to build the current answer.

In [1]:
using System;
using System.Linq;

foreach(int n in new[]{0,4})Console.WriteLine($"sum({n})={SumDown(n)}");
try{SumDown(-1);}catch(ArgumentOutOfRangeException){Console.WriteLine("negative input rejected");}
long SumDown(int n)
{
 if(n<0 || n>100)throw new ArgumentOutOfRangeException(nameof(n),"Teaching demo accepts 0 through 100.");
 return n==0?0:n+SumDown(n-1);
}

sum(0)=0
sum(4)=10
negative input rejected


The contract accepts integers 0 through 100; zero returns zero and negative input is rejected. The cap keeps this linear-depth teaching demo small, rather than promising an unlimited safe recursion depth. For n>0, the nonnegative integer measure n decreases strictly. Assume the smaller call returns the correct sum of 1 through n−1; adding n gives the required sum through n. This is an induction argument: direct base case, smaller correct result, correct combination.

**Common mistake.** Writing the recursive case as `n + SumDown(n)` instead of `SumDown(n - 1)` compiles and never reaches the base case; the calls pile up until the runtime stack is exhausted. Every recursive call must pass a strictly smaller argument.


```{index} Tracing the Call Stack
```

## Tracing the Call Stack




```{index} Tracing the Call Stack
```

The call stack stores each active method call until that call returns. A trace can make the order of calls and returns visible.

In [1]:
using System;
using System.Linq;

Console.WriteLine($"Result: {Factorial(4)}");
Console.WriteLine($"0!={Factorial(0)}");
try{Factorial(-1);}catch(ArgumentOutOfRangeException){Console.WriteLine("negative factorial rejected");}
long Factorial(int n)
{
 if(n<0 || n>20)throw new ArgumentOutOfRangeException(nameof(n),"long factorial accepts 0 through 20.");
 Console.WriteLine($"call Factorial({n})");
 if(n<=1){Console.WriteLine($"return 1 from Factorial({n})");return 1;}
 long smaller=Factorial(n-1);
 long result=checked(n*smaller);
 Console.WriteLine($"return {result} from Factorial({n})");
 return result;
}

call Factorial(4)
call Factorial(3)
call Factorial(2)
call Factorial(1)
return 1 from Factorial(1)
return 2 from Factorial(2)
return 6 from Factorial(3)
return 24 from Factorial(4)
Result: 24
call Factorial(0)
return 1 from Factorial(0)
0!=1
negative factorial rejected


Each frame retains its own n and resumes after the smaller call returns. For 4!, calls enter with 4,3,2,1; returns deliver 1,2,6,24. The suspended multiplication is work done during unwinding. Zero factorial is one. A long holds 20! but not 21!, so the public contract rejects larger inputs; checked arithmetic makes accidental overflow visible. A mathematically terminating recursion can still exhaust the runtime stack.

**Common mistake.** Reading the trace as if the multiplications happened on the way down. Each `return` line appears only after the deeper call has finished, so `4 * 6` is computed last, when `Factorial(4)` resumes; that is why the result lines come out in the opposite order from the call lines.


```{index} Recursive Structure
```

## Recursive Structure




```{index} tree; recursive structure
```


Trees are recursive because each child is the root of a smaller tree. That means many tree algorithms follow the same pattern:

1. Stop if the current node is `null`.
2. Do some work at the current node.
3. Recur into one or more child subtrees.

This is not recursion for its own sake. It mirrors the structure of the data.


In [1]:
using System;

var root = new TreeNode(8);
root.Left = new TreeNode(3);
root.Right = new TreeNode(10);
root.Left.Left = new TreeNode(1);
root.Left.Right = new TreeNode(6);

Console.WriteLine($"Node count: {Count(root)}");
Console.WriteLine($"Leaf count: {CountLeaves(root)}");

int Count(TreeNode? node)
{
    if (node is null)
    {
        return 0;
    }

    return 1 + Count(node.Left) + Count(node.Right);
}

int CountLeaves(TreeNode? node)
{
    if (node is null)
    {
        return 0;
    }

    if (node.Left is null && node.Right is null)
    {
        return 1;
    }

    return CountLeaves(node.Left) + CountLeaves(node.Right);
}

class TreeNode
{
    public int Value { get; }
    public TreeNode? Left { get; set; }
    public TreeNode? Right { get; set; }

    public TreeNode(int value) => Value = value;
}


Node count: 5
Leaf count: 3


`Count` treats the current node as one item plus the size of each child subtree. `CountLeaves` changes the base case: a node with no children contributes one leaf.


**Common mistake.** Forgetting the `null` base case and testing `node.Left is null` only. A call on an empty subtree then dereferences `null`, and the leaf count is wrong even when it does not crash, because `Count` would skip the missing-child contribution of zero.


The recursion needs two guarantees. First, a null reference represents an empty subtree and contributes zero. Second, each recursive call receives a smaller subtree. In a finite tree with no cycles, repeated calls eventually reach null. A cycle would break this termination argument.

For the subtree rooted at 3, `Count(3)` waits for `Count(1)` and `Count(6)`. Both return one, so the call returns `1 + 1 + 1 = 3`. The call at 8 combines that result with the one node rooted at 10 and returns five. Each return summarizes a subtree; it does not count the same node again.

| Call | Left result | Right result | Returned count |
| --- | ---: | ---: | ---: |
| Count(1) | 0 | 0 | 1 |
| Count(6) | 0 | 0 | 1 |
| Count(3) | 1 | 1 | 3 |
| Count(10) | 0 | 0 | 1 |
| Count(8) | 3 | 1 | 5 |

`CountLeaves` has a different meaning: an internal node contributes no leaf of its own, while a node with two null children contributes one. That distinction explains why adding one unconditionally would count nodes instead of leaves.

```{index} Guided Practice: Sum the Values
```

### Guided Practice: Sum the Values


Complete `Sum` so it adds every value in the tree, following the same shape as `Count`: a base case for `null` and one recursive call per child. Expected output: `28`.


In [ ]:
using System;

var root = new SumNode(8);
root.Left = new SumNode(3);
root.Right = new SumNode(10);
root.Left.Left = new SumNode(1);
root.Left.Right = new SumNode(6);

Console.WriteLine(Sum(root));

static int Sum(SumNode? node)
{
    // Your code starts here.
    return 0;
    // Your code ends here.
}

class SumNode
{
    public int Value { get; }
    public SumNode? Left { get; set; }
    public SumNode? Right { get; set; }
    public SumNode(int value) => Value = value;
}


In [ ]:
using System;

// Solution
var root = new SumNode(8);
root.Left = new SumNode(3);
root.Right = new SumNode(10);
root.Left.Left = new SumNode(1);
root.Left.Right = new SumNode(6);

Console.WriteLine(Sum(root));

static int Sum(SumNode? node)
{
    if (node is null) return 0;
    return node.Value + Sum(node.Left) + Sum(node.Right);
}

class SumNode
{
    public int Value { get; }
    public SumNode? Left { get; set; }
    public SumNode? Right { get; set; }
    public SumNode(int value) => Value = value;
}


`Count` and `Sum` differ in one token: `1 +` becomes `node.Value +`. The recursive structure of the tree dictates the shape of every whole-tree summary; only the combining step changes.


```{index} Complete a Recursive Summary Before Designing a New Query
```

## Complete a Recursive Summary Before Designing a New Query


A tree call returns a summary of its subtree. Counting nodes and counting leaves have different meanings even though both visit the same structure. Before inventing another query, decide what null contributes, what a leaf contributes, and how an internal node combines its children.

Use root 8 with left child 3 and right child 10; node 3 has only a left child 1. For a leaf count, null contributes zero and a real leaf contributes one. An internal node contributes the sum of its child leaf counts, without adding one for itself.

| Subtree root | Is it a leaf? | Left leaf count | Right leaf count | Return |
| --- | --- | ---: | ---: | ---: |
| 1 | Yes | Not needed | Not needed | 1 |
| 3 | No | 1 | 0 | 1 |
| 10 | Yes | Not needed | Not needed | 1 |
| 8 | No | 1 | 1 | 2 |

Guided task: complete the leaf condition and the returned combination in this schematic method before reading the runnable checkpoint.

```text
if (node is null) return 0;
if (node.Left is null /* combine with */ node.Right is null) return 1;
return CountLeaves(node.Left) /* combine */ CountLeaves(node.Right);
```

Both child references must be null for a leaf. A node with just one child remains internal. Adding one at every internal node would turn this calculation into a node count.


In [1]:
using System;
#nullable enable
static int CountLeaves(Node? node)
{
    if (node is null) return 0;
    if (node.Left is null && node.Right is null) return 1;
    return CountLeaves(node.Left) + CountLeaves(node.Right);
}
var root = new Node(8, new Node(3, new Node(1)), new Node(10));
Console.WriteLine($"Leaves: {CountLeaves(root)}");
Console.WriteLine($"Empty: {CountLeaves(null)}");
Console.WriteLine($"Single: {CountLeaves(new Node(4))}");
sealed class Node
{
    public int Value { get; }
    public Node? Left { get; }
    public Node? Right { get; }
    public Node(int value, Node? left = null, Node? right = null)
    { Value = value; Left = left; Right = right; }
}


Leaves: 2
Empty: 0
Single: 1


The missing operators are `&&` and `+`. The empty and singleton cases distinguish two different base meanings. As a variation, give node 10 one child 12: it stops being a leaf but still contributes one leaf through 12, so the total remains two. Give it two children instead and the total becomes three. Derive those returns before running them.

For independent work, design a query that counts nodes with exactly one child. State its contribution for null, a leaf, a one-child node and a two-child node before writing the recursion. Reuse the same return-table method to test the proposed rule. These are content-section reasoning steps; the chapter lab then applies tree operations to its service-desk model.


**Common mistake.** Adding one for the internal node in `CountLeaves`, as `Count` does. That counts every node, not every leaf; the table shows that node 3 and node 8 contribute only what their children return.


```{rubric} Footnotes
```
